In [ ]:
import igraph as ig
import networkx as nx
import time
from collections import defaultdict
from community import community_louvain


# Elegir el archivo a cargar
#wikipedia_link_yi_25k
#archivo_grafo = '/Users/biblioadmin/Library/CloudStorage/OneDrive-UniversidadComplutensedeMadrid(UCM)/Documentos/IFSA23/Special Issue/Redes/out.wikipedia_link_yi'

#slashdot
#archivo_grafo = '/Users/biblioadmin/Library/CloudStorage/OneDrive-UniversidadComplutensedeMadrid(UCM)/Documentos/IFSA23/Special Issue/Redes/out.slashdot'

#out.link-dynamic-simplewiki
#archivo_grafo = '/Users/biblioadmin/Library/CloudStorage/OneDrive-UniversidadComplutensedeMadrid(UCM)/Documentos/IFSA23/Special Issue/Redes/out.link-dynamic-simplewiki'

#out.san-francisco
#archivo_grafo = '/Users/biblioadmin/Library/CloudStorage/OneDrive-UniversidadComplutensedeMadrid(UCM)/Documentos/IFSA23/Special Issue/Redes/out.san-francisco'

#out.wiki_talk_de
#archivo_grafo = '/Users/biblioadmin/Library/CloudStorage/OneDrive-UniversidadComplutensedeMadrid(UCM)/Documentos/IFSA23/Special Issue/Redes/out.wiki_talk_de'


# Crear una lista de aristas con pesos
edges = []
with open(archivo_grafo, 'r') as f:
    for line in f:
        # Filtrar líneas que comienzan con '%' (comentarios)
        if line.startswith('%'):
            continue
        # Dividir cada línea en partes (nodo1, nodo2, [peso opcional])
        parts = line.split()
        
        # Asegurarse de que haya al menos dos nodos (para formar una arista)
        if len(parts) >= 2:
            u, v = parts[0], parts[1]
            weight = 1  # Valor predeterminado del peso

            # Si hay una tercera columna (peso), asignarlo
            if len(parts) == 3:
                try:
                    weight = float(parts[2])  # Convertir el peso a float
                except ValueError:
                    pass  # Si no se puede convertir a float, se usa el peso predeterminado

            # Añadir la arista con el peso correspondiente
            edges.append((u, v, weight))

# Crear el grafo a partir de las aristas
G = nx.DiGraph()  # Grafo dirigido
G.add_weighted_edges_from(edges)

# Verifica si se cargó correctamente el grafo
print(f"Gráfico cargado: {len(G.nodes)} nodos, {len(G.edges)} aristas.")

# Crear diccionarios con los grados ponderados (para grafos dirigidos)
in_degrees_weighted = dict(G.in_degree(weight="weight"))
out_degrees_weighted = dict(G.out_degree(weight="weight"))


In [ ]:
resultados_excel = []  # Lista para almacenar los resultados para el Excel
resultados_grafo = []  # Lista para almacenar los resultados de cada grafo
ganamos = 0  # Variable para contar cuántas veces se ha ganado (mejor modularidad)

# Semilla para reproducibilidad
semilla = 123

# Recorremos de 0 a 100 para ajustar el peso de las aristas
for i in range(101):  # De 0 a 100 valores distintos de alpha
    # Actualizamos los pesos de las aristas con el peso combinado
    for u, v, data in G.edges(data=True):
        # Calculamos el nuevo peso 'weight2' como una mezcla ponderada
        data['weight2'] = (1 - (i / 100)) * data['weight'] + (i / 100) * min(
            in_degrees_weighted.get(u, 0), out_degrees_weighted.get(v, 0)
        )  # weight2 va actualizando los pesos de la matriz de adyacencia en función de la agregación

    # Crear un grafo no dirigido con el nuevo peso
    G_undirected = G.to_undirected()

    # Convertir a igraph para después poder calcular la modularidad dirigida con modularity
    G_ig = ig.Graph(directed=True)

    # Convertir nodos de tipo str a enteros (indexación)
    node_mapping = {node: idx for idx, node in enumerate(sorted(G.nodes))}
    G_ig.add_vertices(len(node_mapping))

    # Convertir las aristas con índices enteros
    edges_ig = [(node_mapping[u], node_mapping[v]) for u, v, _ in G.edges(data=True)]
    weights = [d["weight"] for _, _, d in G.edges(data=True)]

    # Agregar las aristas y pesos al grafo igraph
    G_ig.add_edges(edges_ig)
    G_ig.es["weight"] = weights  ##pesos originales

    # Registrar tiempo de inicio
    start_time = time.time()

    # Detectar comunidades usando Louvain, ojo Louvain siempre con no dirigido y con los pesos actuales dependiendo del valor de alpha
    LouvainMejor = community_louvain.best_partition(G_undirected, weight='weight2', random_state=semilla)

    # Registrar tiempo de fin
    end_time = time.time()  # Solo consideramos el tiempo del algoritmo
    tiempo = end_time - start_time

    # Convertir a lista de comunidades, esto se hace para poder calcular la modularidad dirigida de igraph
    com_dict = defaultdict(list)
    for node, community in LouvainMejor.items():
        com_dict[community].append(node)
    LouvainMejorlist = list(com_dict.values())

    # Mapear nodos a comunidades para igraph
    node_mapping_igraph = {node: idx for idx, community in enumerate(LouvainMejorlist) for node in community}
    partition = [node_mapping_igraph[node] for node in sorted(G.nodes)]

    # Calcular modularidad dirigida en la matriz original
    modularidad_Lo = G_ig.modularity(partition, weights=G_ig.es['weight'])

    # Almacenar resultados
    resultados_grafo.append((modularidad_Lo, partition, tiempo))

    # Guardar los resultados en formato tabular para Excel
    resultados_excel.append({
        "i": i,
        "Modularidad": modularidad_Lo,
        "Partición": str(partition),
        "Tiempo (s)": tiempo
    })

# Identificar el valor de i con la mejor modularidad (para i != 0)
mejor_modularidad = max(resultados_grafo[1:], key=lambda x: x[0])  # Omite el primer valor i=0
mejor_modularidad_value = mejor_modularidad[0]
mejor_i = resultados_grafo.index(mejor_modularidad)

# Marcar "Ganamos" si la mejor modularidad no corresponde a i=0
for resultado in resultados_excel:
    if resultado["i"] == mejor_i and resultado["Modularidad"] == mejor_modularidad_value:
        resultado["Ganamos"] = 1  # Marcamos la fila correspondiente con "Ganamos"
    else:
        resultado["Ganamos"] = 0  # No ganamos si no es la mejor modularidad

# Guardar los resultados en un archivo Excel
df_resultados = pd.DataFrame(resultados_excel)
df_resultados.to_excel("wiki_talk_de.xlsx", index=False)


In [ ]:
import pickle

# Guardar el objeto `resultados_grafo` en un archivo pickle
with open("wiki_talk_de", "wb") as f:
    pickle.dump(df_resultados, f)


In [ ]:
# Cargar el objeto `resultados_grafo` desde el archivo pickle
with open("resultados_grafo.pkl", "rb") as f:
    resultados_grafo_cargado = pickle.load(f)

# Verifica que se cargó correctamente
print(resultados_grafo_cargado)
